# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarali8/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

## 1. Build the feature vector

I generate 2,000 synthetic customers as of a single snapshot date, `observation_date =
2025-01-01`. Every "as of snapshot" field below is deliberately computed only from data up to
that date — except two columns I inject on purpose (`cancellation_reason`,
`total_orders_alltime`) to give Section 3 something real to catch.

Engineered features:
- `tenure_days` — from `signup_date` to `observation_date`.
- `has_orders_90d` — flag, then `avg_order_value_90d` filled with 0 when there were no orders
  (an average of zero orders isn't "missing at random", it's structurally undefined).
- `payment_method` — categorical, ~5% missing, filled with the literal category `"unknown"`
  rather than a mode (a silent mode-fill would hide the fact that "we don't know" is itself
  informative and shouldn't be pretended away).
- `email_opt_in` — boolean with some unknowns, also filled as `"unknown"` category.
- `region` — categorical, one-hot encoded.


In [9]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 2000
observation_date = pd.Timestamp("2025-01-01")

# --- latent "true" propensity to churn (not a feature, used only to generate labels) ---
tenure_days = rng.integers(30, 1000, size=n)
engagement = rng.beta(2, 2, size=n)  # 0..1, higher = more engaged
days_since_last_login = rng.poisson(lam=(1 - engagement) * 25 + 1, size=n)
support_tickets_90d = rng.poisson(lam=(1 - engagement) * 2.5, size=n)
orders_90d = rng.poisson(lam=engagement * 4, size=n)

churn_logit = (
    -1.2
    + 0.05 * days_since_last_login
    + 0.35 * support_tickets_90d
    - 0.6 * orders_90d
    - 0.0015 * tenure_days
    + rng.normal(0, 0.8, size=n)
)
churn_prob = 1 / (1 + np.exp(-churn_logit))
churned_next_30d = (rng.random(n) < churn_prob).astype(int)

# --- observed features (as of observation_date) ---
signup_date = observation_date - pd.to_timedelta(tenure_days, unit="D")

avg_order_value_90d = np.where(
    orders_90d > 0, rng.normal(45, 15, size=n).clip(5, None), np.nan
)

is_premium_member = rng.random(n) < 0.3

regions = np.array(["north", "south", "east", "west"])
region = rng.choice(regions, size=n)

payment_methods = np.array(["card", "paypal", "bank_transfer"])
payment_method = rng.choice(payment_methods, size=n).astype(object)
missing_pay_idx = rng.choice(n, size=int(0.05 * n), replace=False)
payment_method[missing_pay_idx] = np.nan

email_opt_in = rng.random(n) < 0.6
email_opt_in = email_opt_in.astype(object)
missing_email_idx = rng.choice(n, size=int(0.08 * n), replace=False)
email_opt_in[missing_email_idx] = np.nan

discount_used_90d = rng.poisson(lam=0.6, size=n)

# --- deliberate leakage traps (kept in the RAW table on purpose; hunted in Section 3) ---
# (a) label-derived: this field is only ever populated *after* a customer has churned
cancellation_reason = np.full(n, np.nan, dtype=object)
churned_idx = np.where(churned_next_30d == 1)[0]
cancellation_reason[churned_idx] = rng.choice(
    ["price", "service", "competitor", "no_reason_given"], size=len(churned_idx)
)

# (b) temporal leakage: "all-time" total includes orders placed AFTER observation_date,
# and churners systematically place ~0 orders after that date -- so this field
# smuggles in the future outcome under an innocent-looking name.
future_orders = np.where(churned_next_30d == 1, rng.poisson(0.1, n), rng.poisson(3.5, n))
total_orders_alltime = orders_90d + future_orders

raw = pd.DataFrame({
    "customer_id": [f"C{100000+i}" for i in range(n)],
    "observation_date": observation_date,
    "signup_date": signup_date,
    "tenure_days": tenure_days,
    "orders_90d": orders_90d,
    "avg_order_value_90d": avg_order_value_90d,
    "days_since_last_login": days_since_last_login,
    "support_tickets_90d": support_tickets_90d,
    "is_premium_member": is_premium_member,
    "region": region,
    "payment_method": payment_method,
    "email_opt_in": email_opt_in,
    "discount_used_90d": discount_used_90d,
    "cancellation_reason": cancellation_reason,     # leakage trap (a)
    "total_orders_alltime": total_orders_alltime,   # leakage trap (b)
    "churned_next_30d": churned_next_30d,           # label
})

raw.head()


,customer_id,observation_date,signup_date,tenure_days,orders_90d,avg_order_value_90d,days_since_last_login,support_tickets_90d,is_premium_member,region,payment_method,email_opt_in,discount_used_90d,cancellation_reason,total_orders_alltime,churned_next_30d
0,C100000,2025-01-01,2024-09-07,116,0,NaN,16,0,True,west,bank_transfer,False,0,price,0,1
1,C100001,2025-01-01,2022-11-13,780,1,32.945456,14,4,True,east,card,True,1,NaN,5,0
2,C100002,2025-01-01,2023-03-09,664,2,46.969438,15,2,False,west,bank_transfer,False,0,NaN,7,0
3,C100003,2025-01-01,2023-10-04,455,4,39.347203,5,0,False,east,card,True,0,NaN,8,0
4,C100004,2025-01-01,2023-10-09,450,4,60.278635,15,0,True,east,paypal,False,2,NaN,8,0


In [10]:
# --- feature engineering / categorical handling / fills ---
df = raw.copy()

df["has_orders_90d"] = (df["orders_90d"] > 0).astype(int)
df["avg_order_value_90d"] = df["avg_order_value_90d"].fillna(0.0)

df["payment_method"] = df["payment_method"].fillna("unknown")
df["email_opt_in"] = df["email_opt_in"].fillna("unknown").astype(str)

cat_cols = ["region", "payment_method", "email_opt_in"]
df_encoded = pd.get_dummies(df, columns=cat_cols, prefix=cat_cols)

df_encoded["is_premium_member"] = df_encoded["is_premium_member"].astype(int)

feature_vector_preview = df_encoded.drop(
    columns=["customer_id", "observation_date", "signup_date"]
)
feature_vector_preview.head()

,tenure_days,orders_90d,avg_order_value_90d,days_since_last_login,support_tickets_90d,is_premium_member,discount_used_90d,cancellation_reason,total_orders_alltime,churned_next_30d,...,region_north,region_south,region_west,payment_method_bank_transfer,payment_method_card,payment_method_paypal,payment_method_unknown,email_opt_in_False,email_opt_in_True,email_opt_in_unknown
0,116,0,0.000000,16,0,1,0,price,0,1,...,False,False,True,True,False,False,False,True,False,False
1,780,1,32.945456,14,4,1,1,NaN,5,0,...,False,False,False,False,True,False,False,False,True,False
2,664,2,46.969438,15,2,0,0,NaN,7,0,...,False,False,True,True,False,False,False,True,False,False
3,455,4,39.347203,5,0,0,0,NaN,8,0,...,False,False,False,False,True,False,False,False,True,False
4,450,4,60.278635,15,0,1,2,NaN,8,0,...,False,False,False,False,False,True,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available before `observation_date`? |
|---|---|---|---|
| `tenure_days` | Days between signup and observation date | None expected (derived from two always-present dates) | Yes |
| `orders_90d` | Count of orders in the 90 days before observation | None expected (0 is a valid count) | Yes |
| `avg_order_value_90d` | Mean order value over that same 90-day window | Structurally missing when `orders_90d == 0`; filled with 0.0 and covered by `has_orders_90d` flag | Yes |
| `has_orders_90d` | Engineered flag for "placed at least 1 order in 90d" | N/A (derived) | Yes |
| `days_since_last_login` | Recency of last login as of observation date | None expected | Yes |
| `support_tickets_90d` | Count of support tickets opened in the 90-day window | None expected (0 is valid) | Yes |
| `is_premium_member` | Whether the account is on the premium tier | None expected | Yes |
| `region` | Customer's registered region (categorical) | Not present in this synthetic set, but if it were, I'd fill with `"unknown"`, never a mode, since region isn't missing-at-random | Yes |
| `payment_method` | Primary payment method on file (categorical) | ~5% missing; filled with explicit `"unknown"` category (not mode) | Yes |
| `email_opt_in` | Whether the customer opted into marketing email | ~8% missing; filled with explicit `"unknown"` category | Yes |
| `discount_used_90d` | Count of discount codes redeemed in the 90-day window | None expected | Yes |
| `cancellation_reason` | Self-reported reason given at cancellation | Only populated for customers who already churned | **No — only exists after the outcome. Excluded.** |
| `total_orders_alltime` | Sum of orders across the customer's full history | None expected, but the window extends *past* `observation_date` | **No — includes future orders. Excluded.** |
| `customer_id` | Row identifier | N/A | Not a behavioral feature; identifier only |

The two "No" rows are exactly the leakage traps built into Section 1 — flagged here from the
column *definitions* alone, before running any statistics. Section 3 then confirms the same
two columns turn up from the data side.


In [11]:
# Sanity-check the notes above against the actual data: missingness per column,
# and a check that the two "No" columns really do depend on the label / the future.
missing_report = raw.isna().mean().sort_values(ascending=False)
print("Missing rate per raw column:")
print(missing_report[missing_report > 0])

print()
print("cancellation_reason non-null ONLY when churned_next_30d == 1?",
      raw.loc[raw["cancellation_reason"].notna(), "churned_next_30d"].eq(1).all())


Missing rate per raw column:
cancellation_reason    0.8100
avg_order_value_90d    0.2095
email_opt_in           0.0800
payment_method         0.0500
dtype: float64

cancellation_reason non-null ONLY when churned_next_30d == 1? True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 3. The leakage hunt

Three concrete tests, run against the encoded feature table plus the label:

1. **Conditional-missingness test** — does a column's missingness pattern depend on the
   label itself (i.e. it's only ever filled in *after* the outcome is known)?
2. **Single-feature predictive-power test** — fit the label on one feature at a time and
   flag anything with an unreasonably high AUC (a single behavioral signal getting >0.9 AUC
   alone is a red flag, not a good model).
3. **Definition/window check** — for each numeric feature, re-derive by hand whether its
   time window could reach past `observation_date` (this is a manual read of Section 1's
   generation code, not a statistic — some leakage is invisible to correlation tests entirely).


In [12]:
from sklearn.metrics import roc_auc_score

y = raw["churned_next_30d"]

# --- Test 1: conditional missingness vs label ---
print("TEST 1 — conditional missingness")
for col in ["cancellation_reason", "avg_order_value_90d", "payment_method", "email_opt_in"]:
    present = raw[col].notna()
    rate_when_churn = present[y == 1].mean()
    rate_when_not = present[y == 0].mean()
    print(f"  {col:22s} present-rate | churned={rate_when_churn:.2f}  not-churned={rate_when_not:.2f}")
print("  -> cancellation_reason is present for churners only: label-derived. FLAG.")
print()

# --- Test 2: single-feature AUC ---
print("TEST 2 — single-feature AUC (numeric columns only)")
numeric_cols = [
    "tenure_days", "orders_90d", "avg_order_value_90d", "days_since_last_login",
    "support_tickets_90d", "discount_used_90d", "total_orders_alltime",
]
suspects = []
for col in numeric_cols:
    vals = raw[col].fillna(raw[col].median())
    auc = roc_auc_score(y, vals)
    auc = max(auc, 1 - auc)  # direction-agnostic
    flag = "  <-- FLAG (>0.90)" if auc > 0.90 else ""
    if flag:
        suspects.append(col)
    print(f"  {col:22s} AUC={auc:.3f}{flag}")
print(f"  -> suspected leakage columns from AUC alone: {suspects}")
print()

# --- Test 3: manual window check (documents what the stats can't see) ---
print("TEST 3 — window/definition check (manual, by design)")
print("  total_orders_alltime = orders_90d + orders placed AFTER observation_date")
print("  -> by construction this reaches past the prediction moment. FLAG,")
print("     even though its AUC alone might look unremarkable on a different random seed.")


TEST 1 — conditional missingness
  cancellation_reason    present-rate | churned=1.00  not-churned=0.00
  avg_order_value_90d    present-rate | churned=0.62  not-churned=0.83
  payment_method         present-rate | churned=0.94  not-churned=0.95
  email_opt_in           present-rate | churned=0.93  not-churned=0.92
  -> cancellation_reason is present for churners only: label-derived. FLAG.

TEST 2 — single-feature AUC (numeric columns only)
  tenure_days            AUC=0.592
  orders_90d             AUC=0.695
  avg_order_value_90d    AUC=0.507
  days_since_last_login  AUC=0.670
  support_tickets_90d    AUC=0.647
  discount_used_90d      AUC=0.516
  total_orders_alltime   AUC=0.955  <-- FLAG (>0.90)
  -> suspected leakage columns from AUC alone: ['total_orders_alltime']

TEST 3 — window/definition check (manual, by design)
  total_orders_alltime = orders_90d + orders placed AFTER observation_date
  -> by construction this reaches past the prediction moment. FLAG,
     even though its AU

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

| Field | Why excluded |
|---|---|
| `cancellation_reason` | Label leakage — only exists once the customer has already churned; caught by Test 1. |
| `total_orders_alltime` | Temporal leakage — its window extends past `observation_date`, confirmed by Test 2's inflated AUC and Test 3's definition check. |
| `customer_id` | Row identifier, not a behavioral signal; a model could memorize IDs instead of learning patterns, and it carries no generalizable meaning. |
| `signup_date` / `observation_date` (raw timestamps) | Redundant with `tenure_days`; keeping the raw calendar dates risks the model keying off calendar-specific artifacts (e.g. a particular signup cohort) that won't hold up on future dates. |
| Any name, email address, physical address, or free-text notes | Not generated into this dataset at all — by design, this synthetic table carries no direct identifiers or PII, so there was nothing here to exclude at the modeling stage; the discipline was applied at generation time instead. |

No client names, real URLs, or private queries appear anywhere in this notebook — the entire
table is synthetic and generated in-memory.

In [13]:
# Final, clean feature matrix: engineered features only, leakage traps and identifiers removed.
excluded_cols = [
    "customer_id", "observation_date", "signup_date",
    "cancellation_reason", "total_orders_alltime",
]

X_final = df_encoded.drop(columns=excluded_cols)
y_final = X_final.pop("churned_next_30d")

assert not any(c in X_final.columns for c in excluded_cols), "excluded column leaked back in"
assert "cancellation_reason" not in X_final.columns
assert "total_orders_alltime" not in X_final.columns

print("Final feature matrix shape:", X_final.shape)
print("Final columns:")
for c in X_final.columns:
    print(" -", c)


Final feature matrix shape: (2000, 19)
Final columns:
 - tenure_days
 - orders_90d
 - avg_order_value_90d
 - days_since_last_login
 - support_tickets_90d
 - is_premium_member
 - discount_used_90d
 - has_orders_90d
 - region_east
 - region_north
 - region_south
 - region_west
 - payment_method_bank_transfer
 - payment_method_card
 - payment_method_paypal
 - payment_method_unknown
 - email_opt_in_False
 - email_opt_in_True
 - email_opt_in_unknown


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.